# 第15回: マルチエージェント

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cosmac-dev/ai-agent-seminar/blob/main/session15/session15_multi_agent.ipynb)

マルチエージェントは、複数のエージェントに役割を分担し、共通の目的へ向けて協調させる構成。
今回は親エージェントが担当のサブエージェントへ仕事を委譲し、独立した仕事を並行に進め、全結果が揃ってから集約する仕組みを実装する。

---

## 0. 環境準備

In [ ]:
# @markdown 実行環境フラグ: Google Colab で実行する場合は True にする
IS_COLAB = False # @param {type:"boolean"}

print("実行環境:", "Google Colab" if IS_COLAB else "ローカル / Dev Container")

In [ ]:
# @title 必要なパッケージをインストールする
%pip install -q --break-system-packages "langchain>=1.4" "langgraph>=1.2" "langchain-openai>=0.2"

In [ ]:
# @title APIキーの設定
import getpass
import os

if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OPENAI_API_KEY を入力する: ')

print('OpenAI APIキー設定完了' if os.environ.get('OPENAI_API_KEY') else 'OpenAI APIキー未設定')

---

## 1. マルチエージェントの役割分担と協調

分担の単位は、同じ目的を別々の観点から処理する役割。
本回の例では、機能・導入条件・利用者の要望をそれぞれの担当が調べ、親エージェントが全体をまとめる。
役割名だけでなく、使える入力・担当範囲・返す成果物を決める。

マルチエージェントの代表的なパターンを、仕組みと設計時の着眼点で比較する。

| パターン | 仕組み | 設計時の着眼点 | ユースケース |
| --- | --- | --- | --- |
| [Subagents](https://docs.langchain.com/oss/python/langchain/multi-agent/subagents)（Supervisor 型） | 親がサブエージェントを Tool として呼び出し、結果を受け取る | 委譲と集約の責任を親に置き、担当ごとのコンテキストを分離する | 専門家への委任 |
| [Handoffs](https://docs.langchain.com/oss/python/langchain/multi-agent/handoffs) | Tool による状態の更新をきっかけに、担当エージェントや現在の Tool・指示を切り替える | 前提条件を満たした後に、次の担当や能力へ進む | 状況に応じた担当者の交代 |
| [Router](https://docs.langchain.com/oss/python/langchain/multi-agent/router) | 入力を分類して一つ以上の担当へ振り分け、結果をまとめる | 入力に応じた担当選択。複数の担当への並行分岐も可能 | 入力の種類に応じた専門担当への振り分け |

### 単独で進めるか、分担するか

複雑な仕事でも、適切な Tool と指示を持つ単独エージェントで十分な場合がある。

分担や情報の切り替えを検討する場面:

- Tool が増え、どれを使うかの判断が難しくなる。
- 専門知識や長い指示を一度に渡すと、必要な情報を扱いにくくなる。
- 独立した仕事を並行に進めたい。
- 担当ごとの能力を別々に開発・保守したい。

引き継ぎや集約にも負担がある。エージェントの数を増やすこと自体を目的にせず、何を分ける必要があるかを先に決める。

### 本回の構成との対応

本回は Subagents（Supervisor 型）で実装する。
親エージェントは3つの担当のサブエージェントを Tool として持ち、どの担当に何を依頼するかを LLM が判断する。
サブエージェントは自分の担当範囲の資料だけを受け取り、出典ID付きの引用を返す。

担当の選択は親の LLM に任せるが、本回の依頼では3担当すべてが必要になる。
入力を分類して担当を選ぶ Router や、担当を切り替える Handoffs は実装しない。

### fan-out / fan-in

独立した仕事を複数の担当へ配り、その結果を一つにまとめる協調パターンがfan-out / fan-in。

- **fan-out**: 共通の入力から、複数の独立した仕事へ分岐する。
- **fan-in**: 必要な結果が揃うまで待ち、後続の処理へ合流する。

本回では、親エージェントが機能・導入条件・利用者の要望の3担当を1回の応答でまとめて呼び出し、全担当の結果が揃ってから集約する。
`create_agent` で作ったエージェントは、1回の応答に含まれる複数の Tool 呼び出しを並行に実行する。

[![](https://mermaid.ink/img/pako:eNqNkstKw0AUhl8lnHVaWxubNogbxYUoaNWNSRexGdtCm5EhwUstmNZoQRFv4ELrBbVWFyJ4QbTqw0wnrW9hmqooCDqr-YbvP_wDJwcJrCGQYCqDZxIplRjcYEzROfeMyPWX8ttRLc75fD3csNysXNJClRZrtPBAC-e0eEOLpXjbHW45C2MYZ9hmjVo3bOWRWrsLXL_sXBw2i8_Oqs2et_-QYzK7Xmf2mVM-rj_d_ysyLrPSRWOn2ly0qXXVrFjO_t6PYL9XfkBmdrVxt-Ec7NPCFntx5dMPIdYW2jD-HQY8GPr939Ra654kHT1uEWa7z1v11zK1TqhVedtbbtwuxYGHJElrIBnERDxkEcmqLYRca7oCRgplkQKSeyVIM2d9CZzBRAFFz7vRaVWfwDj7mSbYTKY-wZzWVAP1pdUkUb8M1TTw6Jye-EogXUOkF5u6AVIw7E0EKQezIHWG_VFBDAWESCQYCkYFgYc5kKIRf0joDITFaCAoil1hIc_DvFch4I-IXTwgLW1gMtTeFm9p8u_i8fJn?type=png)](https://mermaid.live/edit#pako:eNqNkl9r2lAYh79KeK-jMzUaDaU3K15Ihc6tN0u8SM3xD2iOnCWsnQqLNqvQMTZb2EXr1rLV6S5Kwa2UTt2HOZ5ov8ViXKWDwXauznN4fi-_A28VslhHIEOuhJ9nCxoxuY20anDeeaRMxp3b02GGCwTWuE1l1v1KGz3aHNLGNW18oc0BbbYyC3dz7tSeYFxi74bUHrD9G2q_r3EJxe1_nDVH7oHDRof_kNMKu3zDnHO3czb5cfVfkS2FtfrTo97spUPti1nXdk-O_wgm_PJJhTm96fe37ocT2mizsSd__i2kF8ICtu5D0ofU3_9N7der2-TBmleEOd5ze_KzQ-1P1O7eHr-aftvLAA95UtRBNomFeCgjUtbmCNX5dBXMAiojFWTvSpBu7QSyuISJCqpR96IVzXiKcfkuTbCVL4Cc00rPPLIqumai9aKWJ9pS0SwTP941sssIMnREHmLLMEEWBH8kyFXYAXklGoyLUjgkxmJCWIiLIg-7IMdjwbC4EopK8ZAgSZGoWOfhhd8hFIxJER6QXjQxSS3Wxd-a-i_TKPKt)

**親エージェントからの委譲と結果の合流**

### A2Aとの関係

A2Aは、独立したエージェント間で依頼・状態・成果物を交換する通信の規約。
マルチエージェントは、誰に何を任せ、どの順序で協調させるかという構成。
A2Aはその通信方式の一つ。同一プロセス内の関数やグラフでも、マルチエージェントの協調構造を作れる。
第14回のOrchestratorも委譲と集約を担ったが、委譲の流れはPlannerの計画と検索に従うコードで決めていた。
A2Aではこのような内部構造は外から見えず、依頼に応じる相手として扱われる。

---

## 2. 担当の入力と成果物を定義する

架空の製品について、仕様書・導入ガイド・利用者アンケートを確認する。
利用者の要望は、実装済みの機能の証拠にはならない。

受け渡しの形式:

- `Evidence`: 出典ID（資料と文の番号）、資料名、原文の一文。
- `Assignment`: 担当の識別子、表示名、渡す資料名、指示。
- `SubagentOutput`: サブエージェントが返す、出典ID付きの引用（`Finding`）の一覧。
- `WorkerResult`: 検査を通った引用と、除外した理由。
- `FinalReport`: 親エージェントが返す、担当をまたいだ所見と根拠の項目ID。

`Evidence` と `Assignment` は `frozen=True` の dataclass で、実行中に変更しない。
LLM が返す `SubagentOutput` と `FinalReport` は Pydantic のモデルで定義し、`create_agent` の `response_format` に渡して構造化出力にする。

In [ ]:
import json
import time
from dataclasses import dataclass

from pydantic import BaseModel, Field

@dataclass(frozen=True)
class Evidence:
    source_id: str
    document: str
    text: str

@dataclass(frozen=True)
class Assignment:
    key: str
    title: str
    documents: tuple[str, ...]
    instruction: str

class Finding(BaseModel):
    source_id: str = Field(description="引用した資料の出典ID")
    quote: str = Field(description="出典IDの文を一字一句そのまま写した引用")

class SubagentOutput(BaseModel):
    findings: list[Finding] = Field(description="担当範囲に該当する引用。該当がなければ空")

@dataclass(frozen=True)
class WorkerResult:
    assignment_key: str
    findings: tuple[Finding, ...]
    rejected: tuple[str, ...]

    def items(self) -> dict[str, Finding]:
        # 親エージェントが参照する項目ID（例: features-1）を振る。
        return {f"{self.assignment_key}-{i}": finding
                for i, finding in enumerate(self.findings, 1)}

class Note(BaseModel):
    text: str = Field(description="担当をまたいで注意すべき点")
    finding_ids: list[str] = Field(description="根拠とした項目ID。担当が返したIDだけを使う")

class FinalReport(BaseModel):
    notes: list[Note]

@dataclass(frozen=True)
class Event:
    actor: str
    phase: str
    elapsed: float

@dataclass(frozen=True)
class WorkflowRun:
    results: tuple[WorkerResult, ...]
    final: FinalReport
    report: str
    events: tuple[Event, ...]
    messages: tuple

### 固定資料と担当

資料は一文ずつ出典IDを付けて管理する（例: `S1-3` は製品仕様書の3文目）。
担当ごとに、渡す資料を `documents` で決める。サブエージェントは他の資料を受け取らない。
導入条件担当は、仕様書と導入ガイドの両方を受け取る。仕様書には機能の説明と利用の前提条件が混在し、アンケートには感想と要望が混在する。担当ごとに引用する文を選ぶ必要がある。
`instruction` は担当サブエージェントへの役割指示になる。

In [ ]:
SAMPLE_EVIDENCE = (
    Evidence("S1-1", "製品仕様書", "製品はCSVファイルのインポートに対応する。"),
    Evidence("S1-2", "製品仕様書", "インポートできるCSVの文字コードはUTF-8のみ。"),
    Evidence("S1-3", "製品仕様書", "オフライン利用には対応しない。"),
    Evidence("S1-4", "製品仕様書", "データはクラウド上に保存される。"),
    Evidence("S1-5", "製品仕様書", "利用には最新版のWebブラウザが必要。"),
    Evidence("S2-1", "導入ガイド", "導入には管理者による初期設定が必要。"),
    Evidence("S2-2", "導入ガイド", "既存データはUTF-8のCSVに変換してから取り込む。"),
    Evidence("S2-3", "導入ガイド", "社内ネットワークから外部への443番ポートの通信を許可する。"),
    Evidence("S3-1", "利用者アンケート", "CSVの取り込み自体は便利。"),
    Evidence("S3-2", "利用者アンケート", "出張先などオフラインの環境でも使いたい。"),
    Evidence("S3-3", "利用者アンケート", "CSVの取り込み手順をもっと分かりやすくしてほしい。"),
    Evidence("S3-4", "利用者アンケート", "Shift_JISのCSVもそのまま取り込めるとよい。"),
)

ASSIGNMENTS = (
    Assignment("features", "機能", ("製品仕様書",),
               "製品が対応する機能と、対応しない機能。利用の前提条件は含めない。"),
    Assignment("requirements", "導入条件", ("製品仕様書", "導入ガイド"),
               "導入する側が事前に満たす条件や作業。製品の機能の説明は含めない。"),
    Assignment("feedback", "利用者の要望", ("利用者アンケート",),
               "利用者が変更や追加を求めている点。感想は含めない。"),
)

In [ ]:
for assignment in ASSIGNMENTS:
    print(assignment.key, assignment.documents, "→", assignment.instruction)
print("資料の数:", len(SAMPLE_EVIDENCE))

---

## 3. サブエージェントと親エージェントを作る

### サブエージェント

各担当は `create_agent` で作るサブエージェント。担当ごとの役割指示と、担当範囲の資料だけを受け取る。
親エージェントの会話履歴や他の担当の資料は渡さない。これがサブエージェントによるコンテキストの分離。
応答は `response_format` に渡した `SubagentOutput` の形で、`structured_response` から受け取る。

In [ ]:
# @title 使用するモデルを選ぶ
from langchain_openai import ChatOpenAI

model_id = 'gpt-5.4-mini' # @param ['gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-5.6-luna', 'gpt-5.5', 'gpt-5.4', 'gpt-5.4-mini', 'gpt-5.4-nano']
model = ChatOpenAI(model=model_id)

In [ ]:
from langchain.agents import create_agent
from langchain_core.language_models import BaseChatModel

SUBAGENT_PROMPT = """あなたは「{title}」担当のサブエージェント。担当範囲: {instruction}
渡された資料の各文が担当範囲に当てはまるかを判断し、当てはまる文だけを引用する。
引用は各行の文を一字一句変えずに写し（行頭の出典IDは含めない）、その行の出典IDを付ける。
該当する記載がなければ findings を空にする。"""

def format_documents(evidence: tuple[Evidence, ...],
                     documents: tuple[str, ...]) -> str:
    lines = []
    for document in documents:
        lines.append(f"## {document}")
        lines += [f"[{item.source_id}] {item.text}"
                  for item in evidence if item.document == document]
    return "\n".join(lines)

async def run_subagent(model: BaseChatModel, assignment: Assignment, request: str,
                       evidence: tuple[Evidence, ...]) -> SubagentOutput:
    subagent = create_agent(
        model,
        system_prompt=SUBAGENT_PROMPT.format(
            title=assignment.title, instruction=assignment.instruction),
        response_format=SubagentOutput,
        name=f"{assignment.key}_agent",
    )
    documents = format_documents(evidence, assignment.documents)
    content = f"依頼: {request}\n\n資料:\n{documents}"
    state = await subagent.ainvoke({"messages": [{"role": "user", "content": content}]})
    return state["structured_response"]

In [ ]:
one_output = await run_subagent(model, ASSIGNMENTS[2], "利用者の要望を確認する",
                                SAMPLE_EVIDENCE)
for finding in one_output.findings:
    print(f"[{finding.source_id}] {finding.quote}")

### 引用を検査する

LLM の応答は、指示どおりの引用になっているとは限らない。サブエージェントの結果を親へ渡す前に、コードで検査する。

- 出典IDが、その担当に渡した資料のものか。
- 引用が、その出典IDの原文と一致するか。

検査を通らない引用は除外し、理由を `rejected` に残す。
原文との一致は引用の検査であり、言い換えや要約の意味的な正しさの検査ではない。

In [ ]:
def validate_findings(assignment: Assignment,
                      evidence: tuple[Evidence, ...],
                      output: SubagentOutput) -> WorkerResult:
    sources = {item.source_id: item for item in evidence
               if item.document in assignment.documents}
    accepted, rejected = [], []
    for finding in output.findings:
        source = sources.get(finding.source_id)
        quote = finding.quote.strip()
        if source is None:
            rejected.append(f"担当範囲にない出典: {finding.source_id}")
        elif quote != source.text:
            rejected.append(f"原文と一致しない引用: {finding.quote}")
        else:
            accepted.append(Finding(source_id=finding.source_id, quote=quote))
    return WorkerResult(assignment.key, tuple(accepted), tuple(rejected))

### サブエージェントを Tool にする

`make_subagent_tool` は、サブエージェントの呼び出し・検査・記録を一つの Tool にまとめる。
親エージェントに見えるのは Tool の名前・説明・戻り値だけ。資料やサブエージェントの内部の会話は見えない。
戻り値には項目ID付きの引用を入れ、親が所見の根拠を項目IDで示せるようにする。
開始・終了のイベントは、並行に動いたかを観察するために記録する。

In [ ]:
from langchain.tools import tool

def make_subagent_tool(model: BaseChatModel,
                       assignment: Assignment,
                       evidence: tuple[Evidence, ...],
                       results: list[WorkerResult],
                       events: list[Event],
                       started: float):
    def record(phase: str) -> None:
        events.append(Event(assignment.key, phase,
                            round(time.perf_counter() - started, 2)))

    @tool(assignment.key,
          description=f"「{assignment.title}」担当のサブエージェントへ調査を依頼する。"
                      "結果は項目ID付きの引用で返る。")
    async def delegate(request: str) -> str:
        record("start")
        output = await run_subagent(model, assignment, request, evidence)
        result = validate_findings(assignment, evidence, output)
        results.append(result)
        record("end")
        items = [{"id": finding_id, **finding.model_dump()}
                 for finding_id, finding in result.items().items()]
        return json.dumps({"findings": items, "rejected": len(result.rejected)},
                          ensure_ascii=False)

    return delegate

### 親エージェント

親エージェントは担当の Tool だけを持ち、資料を直接は読まない。
どの担当に何を依頼するか、全結果をどうまとめるかを LLM が判断する。
親の最終応答は `FinalReport` で、担当をまたいだ所見と、その根拠の項目IDを返す。

In [ ]:
SUPERVISOR_PROMPT = """あなたは製品調査の取りまとめ役。資料は自分で読まず、担当のサブエージェントへ調査を依頼する。
互いに依存しない担当は、同じ応答の中でまとめて呼び出す。
全担当の結果が揃ったら、複数の担当の結果を突き合わせて分かる注意点だけを notes に書き、根拠の項目IDを付ける。
各担当の結果を言い換えただけの内容は書かない。
項目IDは担当が返したものだけを使う。利用者の要望を実装済みの機能として扱わない。"""

async def run_supervisor(model: BaseChatModel,
                         request: str,
                         evidence: tuple[Evidence, ...] = SAMPLE_EVIDENCE,
                         assignments: tuple[Assignment, ...] = ASSIGNMENTS) -> WorkflowRun:
    results: list[WorkerResult] = []
    events: list[Event] = []
    started = time.perf_counter()
    tools = [make_subagent_tool(model, assignment, evidence, results, events, started)
             for assignment in assignments]
    supervisor = create_agent(model, tools, system_prompt=SUPERVISOR_PROMPT,
                              response_format=FinalReport, name="supervisor")
    state = await supervisor.ainvoke({"messages": [{"role": "user", "content": request}]})
    final = state["structured_response"]
    report = render_report(assignments, tuple(results), final)
    return WorkflowRun(tuple(results), final, report, tuple(events),
                       tuple(state["messages"]))

### レポートを組み立てる

引用部分は、親の文章ではなく、検査済みの担当の結果からコードで組み立てる。完了順ではなく、定義した担当の順序で並べる。
該当する記載がない結果と、担当の結果が届いていない状態は区別する。
親の所見は自由記述のため、引用の検査では正しさを保証できない。根拠の項目IDが実在することだけを確認する。

In [ ]:
def render_report(assignments: tuple[Assignment, ...],
                  results: tuple[WorkerResult, ...],
                  final: FinalReport) -> str:
    # 完了順ではなく、担当の識別子で結果を対応付ける。
    by_key = {result.assignment_key: result for result in results}
    if len(by_key) != len(results):
        raise ValueError("同じ担当の結果が複数ある")
    known_ids = {finding_id for result in results for finding_id in result.items()}
    lines = ["# 製品の確認結果"]
    for assignment in assignments:
        lines.append(f"\n## {assignment.title}")
        result = by_key.get(assignment.key)
        if result is None:
            lines.append("- 担当の結果が届いていない。このレポートは未完成。")
            continue
        if not result.findings:
            lines.append("- 該当する記載なし。未確認事項は補わない。")
        for finding_id, finding in result.items().items():
            lines.append(f"- {finding.quote} [{finding.source_id}] ({finding_id})")
    lines.append("\n## 所見（親エージェントの記述。引用の検査対象外）")
    for note in final.notes:
        unknown = [i for i in note.finding_ids if i not in known_ids]
        if unknown:
            raise ValueError(f"担当が返していない項目IDを根拠にしている: {unknown}")
        evidence_ids = ", ".join(note.finding_ids) or "根拠の項目IDなし"
        lines.append(f"- {note.text} ({evidence_ids})")
    return "\n".join(lines)

---

## 4. 実行と集約を観察する

### 親エージェントに依頼する

In [ ]:
run = await run_supervisor(model, "この製品の機能、導入条件、利用者の要望を確認したい。")
print(run.report)

### fan-out / fan-in を確認する

メッセージの履歴から、親がどの応答でどの担当を呼び出したかを表示する。
1回の応答に複数の Tool 呼び出しがあれば、それが fan-out。親の次の応答は、全ての Tool の結果が揃った後に始まる。これが fan-in。
イベントの時刻からは、担当の処理が重なって進んだことと、完了順が呼び出し順と異なり得ることを確認する。

In [ ]:
from langchain_core.messages import AIMessage, ToolMessage

for message in run.messages:
    if isinstance(message, AIMessage) and message.tool_calls:
        print("親 → 呼び出し:", ", ".join(call["name"] for call in message.tool_calls))
    elif isinstance(message, ToolMessage):
        print("    結果 ←", message.name)
    elif isinstance(message, AIMessage):
        print("親 → 最終応答")

print()
for event in run.events:
    print(f"{event.elapsed:6.2f}s  {event.actor:<12} {event.phase}")

呼び出し方は親の LLM の判断のため、実行ごとに変わり得る。
3担当が別々の応答で呼ばれた場合は、逐次に進んだことになる。メッセージの履歴とイベントの時刻で、どちらになったかを確認する。